In [5]:
import pandas as pd
from IPython.display import Audio, display
import soundfile as sf
from pydub import AudioSegment
import numpy as np
import os, gc , time, json
from io import BytesIO
import json
from sklearn.model_selection import train_test_split


AudioPATH= "./audio"
ModelId1="Qwen/Qwen3-ASR-1.7B"
ModelId2="Qwen/Qwen3-ASR-0.6B "

In [6]:
# !pip install pydub

In [7]:
## Convert refvoice  mp3 file to wave file

In [8]:
from pydub import AudioSegment


def convert_mp3_to_wav(src, dst):
    # Convert the MP3 file to a WAV file
    try:
        sound = AudioSegment.from_mp3(src)
        sound.export(dst, format="wav")
        print(f"Successfully converted {src} to {dst}")
    except Exception as e:
        print(f"An error occurred: {e}")

## Load Cantonese dataset, download from huggingface : a cleaned verison from AlienKevin/mixed_cantonese_and_english_speech:
https://huggingface.co/datasets/AlienKevin/mixed_cantonese_and_english_speech

In [9]:
df = pd.read_parquet("train-00000-of-00001.parquet",  engine='pyarrow')

In [10]:
df.columns

Index(['audio', 'sentence', 'topic'], dtype='object')

In [11]:
df

,audio,sentence,topic
0,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,你知唔知今日嘅temperature會落到低至10度，一定要著厚d嘅clothes啊！,天氣
1,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,我check咗天氣app，佢話今日會有thunderstorm，最好唔好去戶外行嘅。,天氣
2,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,飛機已經被grounded喇，因為typhoon signal已經升到No.8嚟啦。,天氣
3,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,周圍都有well-developed霧，嚴重limit我哋嘅visibility，開車嘅時候...,天氣
4,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,Forecast話晚上嘅weather會變得好cold，會降到freezing point，...,天氣
...,...,...,...
14012,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,雖然香港天氣好hot，但我哋可以想下use less air con，改用風扇。,环境
14013,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,Buy local 嘅商品都可以減少carbon footprint，唔再郵寄長途嘅貨品。,环境
14014,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,用solar power同wind power都唔錯，我哋要追求sustainable嘅生活方式。,环境
14015,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,"Reduce, Reuse, Recycle 呢個三R原則喺日常生活中實踐，即係好好嘅典範。",环境


In [12]:
df["audio"][0]

{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x00\x00\x00\x0f\x00\x00\x03Lavf58.76.100\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xf3X\xc0\x00\x00\x00\x00\x00\x00\x00\x00\x00Info\x00\x00\x00\x0f\x00\x00\x00\xe4\x00\x00`\xe4\x00\x05\x07\t\r\x0f\x11\x13\x17\x19\x1b\x1e!#%(+-0257:<?BDFILNPSVX[^`behjlprtvz|~\x82\x84\x86\x88\x8c\x8e\x90\x92\x96\x98\x9a\x9d\xa0\xa2\xa4\xa7\xaa\xac\xaf\xb1\xb4\xb6\xb9\xbc\xbe\xc1\xc3\xc6\xc8\xcb\xcd\xd0\xd2\xd5\xd7\xda\xdd\xdf\xe1\xe4\xe7\xe9\xeb\xef\xf1\xf3\xf5\xf9\xfb\xfd\x00\x00\x00\x00Lavc58.13\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00$\x04\x15\x00\x00\x00\x00\x00\x00`\xe4\xc0zi\x0b\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xf38\xc4\x00\x00\x00\x03H\x00\x00\x00\x00LAME3.100UUUUUUUUUUUUUUUUUUUUULAME3.100UUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUU\xff\xf38\xc4_\x00\x00\x03H\x00\x00\x00\x00UUUUUUUUUUUUUUUUUUUUUUUUUUUUUULAME3.100UUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUU\xff\xf38\xc4\xa0\x00\x00\x03H\x00\x00\x00\x00U

In [13]:
df["topic"].unique()

array(['天氣', '食物', '旅遊', '娛樂', '體育', '本地時事', '購物', '學習', '工作', '健康同健身',
       '宠物', '科技同新闻', '电影同电视剧', '音乐同艺术', '爱好同兴趣', '历史同文学', '社交媒体同网络文化',
       '环境'], dtype=object)

## Checking the Audio and Text label

In [14]:
def playAudio(raw):
    display(Audio(raw))

In [15]:
index = 3

playAudio(df["audio"][index]["bytes"]) 
print(f"Index : {index}")
print(f"Text : {df["sentence"][index]}")
print(f"File: {df["audio"][index]["path"]}")

Index : 3
Text : 周圍都有well-developed霧，嚴重limit我哋嘅visibility，開車嘅時候要小心啲！
File: 1_4.mp3


## Prepare Audio File for Hong Kong Cantonese (Fine tune) Training Dataset

In [16]:
def generateAudioFile(raw : bytes, fileName: str, format_hint="mp3"):
    # Wrap bytes in memory buffer
    audio_buffer = BytesIO(raw)
    # Try to load (pydub guesses format from content or you can force it)
    # audio = AudioSegment.from_file(audio_buffer, format=format_hint)
    data, samplerate = sf.read(audio_buffer)
    # print(samplerate)
    # Write to new file (format guessed from extension)
    sf.write(fileName, data, samplerate)
    
    print(f"Saved via soundfile → {fileName}")
    
    

In [17]:
generateAudioFile(df["audio"][index]["bytes"], f"{AudioPATH}/{index}.wav")

Saved via soundfile → ./audio/3.wav


In [18]:
numberSample= 2000 #500
for idx in range(numberSample):
    generateAudioFile(df["audio"][idx]["bytes"], f"{AudioPATH}/{idx}.wav")

Saved via soundfile → ./audio/0.wav
Saved via soundfile → ./audio/1.wav
Saved via soundfile → ./audio/2.wav
Saved via soundfile → ./audio/3.wav
Saved via soundfile → ./audio/4.wav
Saved via soundfile → ./audio/5.wav
Saved via soundfile → ./audio/6.wav
Saved via soundfile → ./audio/7.wav
Saved via soundfile → ./audio/8.wav
Saved via soundfile → ./audio/9.wav
Saved via soundfile → ./audio/10.wav
Saved via soundfile → ./audio/11.wav
Saved via soundfile → ./audio/12.wav
Saved via soundfile → ./audio/13.wav
Saved via soundfile → ./audio/14.wav
Saved via soundfile → ./audio/15.wav
Saved via soundfile → ./audio/16.wav
Saved via soundfile → ./audio/17.wav
Saved via soundfile → ./audio/18.wav
Saved via soundfile → ./audio/19.wav
Saved via soundfile → ./audio/20.wav
Saved via soundfile → ./audio/21.wav
Saved via soundfile → ./audio/22.wav
Saved via soundfile → ./audio/23.wav
Saved via soundfile → ./audio/24.wav
Saved via soundfile → ./audio/25.wav
Saved via soundfile → ./audio/26.wav
Saved via s

In [19]:
import random

# Generate a random integer between 1 and 10 (inclusive)
random_integer = random.randint(1, numberSample)
print(random_integer)

1517


In [20]:
# generateAudioFile(df["audio"][random_integer]["bytes"], f"{AudioPATH}/ref_voice/alienkevin.wav")

### Resample ALL Audio file Qwen3 ASR requirement only support 24Khz

In [21]:
import librosa
import soundfile as sf

# # wav, sr = librosa.load(f"{AudioPATH}/2.wav", sr=None)
# # wav, sr = librosa.load(RefVoicePATHWav, sr=None)
# wav , sr = librosa.load(f"{AudioPATH}/ref_voice/alienkevin.wav", sr=None)
# print(f"Original Sample rate: {sr}")

# if sr != 24000:
#     newWav = librosa.resample(wav, orig_sr=sr, target_sr=24000)
#     # Add normalization to prevent clipping
#     newWav = librosa.util.normalize(newWav)
#     sf.write(f"{AudioPATH}/ref_voice/alienkevin.wav", newWav, 24000)
#     # print(f"New Sample rate: {tempSR}")
#     sr = 24000
# else:
#      # Normalize even if the sample rate is correct
#     newWav = librosa.util.normalize(wav)
#     sf.write(audioPath, newWav, targetSR)
#     print(f"Audio File ready support target sample rate: {targetSR}")

In [22]:
wav, sr = librosa.load(f"{AudioPATH}/1.wav", sr=None)
sr

/home/aifather/venv-qwen3asr/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


16000

In [23]:
def batchResample(audioPath, targetSR=24000):
    wav, sr = librosa.load(audioPath, sr=None)
    if sr != 24000:
        newWav = librosa.resample(wav, orig_sr=sr, target_sr=24000)
        # Add normalization to prevent clipping
        newWav = librosa.util.normalize(newWav)
        sf.write(audioPath, newWav, targetSR)
        print(f"Saved resample soundfile → {audioPath}")
    else:
        # Normalize even if the sample rate is correct
        newWav = librosa.util.normalize(wav)
        sf.write(audioPath, newWav, targetSR)
        print(f"Audio File ready support target sample rate: {targetSR}")
    

In [24]:
# numberSample= 500
# for idx in range(numberSample):
#     batchResample(f"{AudioPATH}/{idx}.wav", targetSR=24000)

In [25]:
# extract Dataframe

In [26]:
selectedDF =df[:numberSample].copy()

In [27]:
selectedDF

,audio,sentence,topic
0,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,你知唔知今日嘅temperature會落到低至10度，一定要著厚d嘅clothes啊！,天氣
1,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,我check咗天氣app，佢話今日會有thunderstorm，最好唔好去戶外行嘅。,天氣
2,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,飛機已經被grounded喇，因為typhoon signal已經升到No.8嚟啦。,天氣
3,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,周圍都有well-developed霧，嚴重limit我哋嘅visibility，開車嘅時候...,天氣
4,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,Forecast話晚上嘅weather會變得好cold，會降到freezing point，...,天氣
...,...,...,...
1995,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,你嘅work attitude十分影响你嘅career development，要常常che...,工作
1996,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,有時候，唔係要work harder，而係要work smarter。,工作
1997,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,Job satisfaction同money唔一定要相反，有時你最享受嘅work可能就係你最赚嘅。,工作
1998,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,睇得太远可能會miss住身邊嘅opportunity，即使個目标未到，都要appreciat...,工作


### Generate JSONL file for prepare Training data

In [28]:
def generateJSONFile(dataFrame, audioDirectory, outputFile="train.jsonl"):
    with open(outputFile, "w", encoding="utf-8") as f:
        for i, row in dataFrame.iterrows():
            obj = {
                "audio": f"{audioDirectory}/{i}.wav",
                "text": f"language Cantonese<asr_text>{row['sentence']}",   # map sentence -> text
            }
            # ensure_ascii=False keeps Chinese characters, etc.
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [29]:
generateJSONFile(selectedDF, AudioPATH, outputFile="train_raw.jsonl")

In [56]:
def split_jsonl_dataset(
    input_file: str = "train_raw.jsonl",
    val_size: float = 0.2,
    random_state: int = 42,
    output_dir: str = ".",
    train_output_name: str = "train.jsonl",
    val_output_name: str = "val.jsonl"
) -> tuple[list[dict], list[dict]]:
    """
    Splits a JSONL dataset file into training and validation sets.
    
    Args:
        input_file (str): Path to your raw JSONL file (default: train_raw.jsonl).
        val_size (float): Percentage of the data to use for the validation/test set 
                          (e.g. 0.2 = 20%). Must be between 0 and 1.
        random_state (int): Random seed for reproducible splits (default: 42).
        output_dir (str): Directory where train.jsonl and val.jsonl will be saved.
        train_output_name (str): Filename for the training set.
        val_output_name (str): Filename for the validation set.
    
    Returns:
        tuple: (train_data, val_data) — lists of dictionaries for further use if needed.
    
    Example usage:
        train_data, val_data = split_jsonl_dataset(val_size=0.15)
        # or
        split_jsonl_dataset("train_raw.jsonl", val_size=0.2, output_dir="./data_split")
    """
    # 1. Load the entire JSONL file
    with open(input_file, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f]

    print(f"Loaded {len(data)} samples from {input_file}")

    # 2. Split into train + validation (sklearn handles shuffling automatically)
    train_data, val_data = train_test_split(
        data,
        test_size=val_size,
        random_state=random_state
    )

    # 3. Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # 4. Save training set
    train_path = os.path.join(output_dir, train_output_name)
    with open(train_path, "w", encoding="utf-8") as f:
        for item in train_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    # 5. Save validation set
    val_path = os.path.join(output_dir, val_output_name)
    with open(val_path, "w", encoding="utf-8") as f:
        for item in val_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"✅ Split complete!")
    print(f"   • Training samples : {len(train_data):,}")
    print(f"   • Validation samples: {len(val_data):,}")
    print(f"   • Files saved → {train_path}")
    print(f"   • Files saved → {val_path}")

    return train_data, val_data

In [57]:
# Just run this in your script/notebook
split_jsonl_dataset(val_size=0.3)                    # 20% validation
# or
# split_jsonl_dataset(val_size=0.15, output_dir="./my_split")  # 15% validation

Loaded 2000 samples from train_raw.jsonl
✅ Split complete!
   • Training samples : 1,400
   • Validation samples: 600
   • Files saved → ./train.jsonl
   • Files saved → ./val.jsonl


([{'audio': './audio/836.wav',
   'text': 'language Cantonese<asr_text>前晚我試咗嚟自Thailand嘅tom yum soup，味道真係足夠spicy同refreshing，正宗嘅泰國菜味道！'},
  {'audio': './audio/575.wav',
   'text': 'language Cantonese<asr_text>我琴日去左個gallery，有好多modern art展品，雖然有啲我都唔知佢想表達咩，但係我都鐘意去思考同欣賞佢哋！'},
  {'audio': './audio/557.wav',
   'text': 'language Cantonese<asr_text>昨晚上我落廚，整咗煎鴨胸，搭配fresh嘅蔬菜同薯蓉，持平講，冇得嘢比自家煮嘅飯更好味！'},
  {'audio': './audio/1235.wav',
   'text': 'language Cantonese<asr_text>進行terminal射擊運動幫助我哋improve concentration，亦都需要穩定嘅hand control。'},
  {'audio': './audio/1360.wav',
   'text': 'language Cantonese<asr_text>Teamwork係件好事，大家都需要cooperate，咁先能達到the best results。'},
  {'audio': './audio/1347.wav',
   'text': 'language Cantonese<asr_text>我喺Apple store買咗新款iPhone，感覺好好，用得都好開心咁！'},
  {'audio': './audio/1161.wav',
   'text': 'language Cantonese<asr_text>Sogo嘅cosmetic section真係好吸引，各種品牌都有，仲可以試用下先決定買邊款。'},
  {'audio': './audio/1442.wav',
   'text': 'language Cantonese<asr_text>無論考試嘅result如何，最重要嘅係你吸收咗幾多knowledge。'},
  

In [25]:
# trainDF = pd.read_json("train_raw.jsonl", lines=True)

# print(trainDF.head())
# print(trainDF.columns)  # Should show: ['audio', 'text', 'ref_audio']

In [26]:
# trainDF

In [27]:
# trainDF["ref_audio"][0]

### # Download model into local Directory

In [28]:
# Using huggingface-cli (install if needed: pip install -U huggingface_hub[cli])

# !huggingface-cli download Qwen/Qwen3-ASR-1.7B --local-dir ./Qwen3-ASR-1.7B
!huggingface-cli download Qwen/Qwen3-ASR-0.6B --local-dir ./Qwen3-ASR-0.6B

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 10 files:   0%|                                 | 0/10 [00:00<?, ?it/s]Downloading 'model.safetensors' to 'Qwen3-ASR-0.6B/.cache/huggingface/download/xGOKKLRSlIhH692hSVvI1-gpoa8=.79d6cbd4c98c7bbffe9db2edac07f56cd6637d0d5944b27f6c2b8353840323ea.incomplete'

model.safetensors:   0%|                            | 0.00/1.88G [00:00<?, ?B/s]Downloading 'chat_template.json' to 'Qwen3-ASR-0.6B/.cache/huggingface/download/BqEbVjUtvDkrcHAfaoo906xJg7Y=.c44736493efd71ec96218cc626904698cdb13235.incomplete'


preprocessor_config.json: 100%|████████████████| 330/330 [00:00<00:00, 1.32MB/s]


Download complete. Moving file to Qwen3-ASR-0.6B/preprocessor_config.json
.gitattributes: 0.00B [00:00, ?B/s]


chat_template.json: 0.00B [00:00, ?B/s]



generation_config.json: 100%|███████████████████| 142/142 [00:00<00:00, 615kB/s]
chat_template.json: 1.16kB [00:00, 470kB/s]
Download complete. Moving file to Qwen3-ASR-0

# Run Prepare data script from Qwen3 fine turning python script)

In [32]:
%%time 
# ## Support generate both train and evaluate 
# !python prepare_train_evaluate_data.py \
# --device cuda:0 \
# --input_jsonl train_raw.jsonl \
# --output_train_jsonl train_prepared.jsonl \
# --output_eval_jsonl eval_prepared.jsonl \
# --eval_ratio 0.1 \
# --speaker_name hk_cantonese_speaker
# !python prepare_train_dataset.py

CPU times: user 11 μs, sys: 2 μs, total: 13 μs
Wall time: 25.7 μs


# Run SFT using the prepared JSONL:
#### select batch size depend on your gpu memory: minimize at least GPU 12GB VRAM, otherwise error Out-of-Memory (OOM)
#### batch _size =1 or 2 for 12GB (3080 GPU),  batch_size =4 for 16GB GPU VRAM

### MLflow Trace support version

In [30]:
%%time
# !python sft_12hz_mlflow.py \
#   --init_model_path ./Qwen3-TTS-12Hz-0.6B-Base \
#   --output_model_path output \
#   --train_jsonl train_prepared.jsonl \
#   --batch_size 2 \
#   --lr 1e-5 \
#   --num_epochs 20 \
#   --gradient_accumulation_steps 8 \
#   --speaker_name speaker_test \
#   --mlflow_tracking_uri http://localhost:5000 


CPU times: user 11 μs, sys: 2 μs, total: 13 μs
Wall time: 25.5 μs


In [50]:
# upload final model to mlflow
import mlflow
# mlflow.log_artifacts("output/checkpoint-epoch-3", artifact_path="final_checkpoint")


## Test Qwen3 Pre-train model performance

In [74]:
%%time
!python evaluate_qwen3_asr.py \
  --checkpoint_dir ./Qwen3-ASR-0.6B \
  --test_jsonl  val.jsonl \
  --output_dir ./asr_evaluation_results_pretrained \
  --language Cantonese \
  --max_samples 500


Loading Qwen3-ASR from: ./Qwen3-ASR-0.6B
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
✅ Loaded 500 test samples
🚀 Starting evaluation with MANUAL WER/CER (custom tokenizer)...

Evaluating: 100%|█████████████████████████| 500/500 [12:45<00:00,  1.53s/sample]

✅ EVALUATION COMPLETE — MANUAL WER/CER with custom tokenizer
Checkpoint       : ./Qwen3-ASR-0.6B
Test samples     : 500
Avg CER          : 0.2664
Avg WER          : 0.5552   ← now accurate for code-mixing
Avg RTF          : 0.2180
Results saved to : ./asr_evaluation_results_pretrained/results.csv
CPU times: user 12.8 s, sys: 2.26 s, total: 15 s
Wall time: 12min 56s


## Read Result

In [76]:
evalResultFile= "./asr_evaluation_results_pretrained/results.csv"

In [77]:
evalResultDF = pd.read_csv(evalResultFile)
evalResultDF

,index,ref_text,pred_text,cer,wer,rtf,audio_duration_sec,inference_time_sec,ref_words_array,pred_words_array,...,wer_insertions,wer_errors_(S+D+I),ref_chars_array,pred_chars_array,ref_chars,cer_substitutions,cer_deletions,cer_insertions,cer_errors_(S+D+I),audio
0,0,做yoga可以改善你的flexibility，同時亦能提升你的inner peace和focus.,做 yoga 可以改善你嘅 flexibility，同時亦能提升你的 inner peace...,0.1633,0.2000,0.2415,9.899,2.3907,"做, yoga, 可以改善你的, flexibility, ，同時亦能提升你的, inner...","做, yoga, 可以改善你嘅, flexibility, ，同時亦能提升你的, inner...",...,0,2,"做, y, o, g, a, 可, 以, 改, 善, 你, 的, f, l, e, x, i...","做, , y, o, g, a, , 可, 以, 改, 善, 你, 嘅, , f, l...",49,2,0,6,8,./audio/1860.wav
1,1,每日都要set新嘅goals，並keep track嘅progress，咁先可以開展自己嘅p...,每日都要set新嘅goals並keep track嘅progress，咁先可以開展自己嘅po...,0.0182,0.0833,0.2321,6.123,1.4212,"每日都要, set, 新嘅, goals, ，並, keep, track, 嘅, prog...","每日都要, set, 新嘅, goals, 並, keep, track, 嘅, progr...",...,0,1,"每, 日, 都, 要, s, e, t, 新, 嘅, g, o, a, l, s, ，, 並...","每, 日, 都, 要, s, e, t, 新, 嘅, g, o, a, l, s, 並, k...",55,0,1,0,1,./audio/353.wav
2,2,香港嘅public transport system 需要更多嘅improvements去減...,但public transport system需要更多嘅improvement，係減逼co...,0.1207,0.3333,0.3150,3.627,1.1423,"香港嘅, public, transport, system, 需要更多嘅, improve...","但, public, transport, system, 需要更多嘅, improveme...",...,0,3,"香, 港, 嘅, p, u, b, l, i, c, , t, r, a, n, s, p...","但, p, u, b, l, i, c, , t, r, a, n, s, p, o, r...",58,4,3,0,7,./audio/1333.wav
3,3,透過學習，我哋可以empower自己去迎接新嘅challenges。,透過學習，我哋可以empower自己去迎接新嘅challenge。,0.0294,0.2000,0.2933,3.520,1.0323,"透過學習，我哋可以, empower, 自己去迎接新嘅, challenges, 。","透過學習，我哋可以, empower, 自己去迎接新嘅, challenge, 。",...,0,1,"透, 過, 學, 習, ，, 我, 哋, 可, 以, e, m, p, o, w, e, r...","透, 過, 學, 習, ，, 我, 哋, 可, 以, e, m, p, o, w, e, r...",34,0,1,0,1,./audio/905.wav
4,4,趁热食意粉，搞两粒鸡蛋，runny yolk滴落去，滑inh到can't stop！,親要食意粉，攪兩碌雞蛋，rainy york滴落去，或inch到count stop。,0.3571,0.7778,0.2419,6.677,1.6152,"趁热食意粉，搞两粒鸡蛋，, runny, yolk, 滴落去，滑, inh, 到, can'...","親要食意粉，攪兩碌雞蛋，, rainy, york, 滴落去，或, inch, 到, cou...",...,0,7,"趁, 热, 食, 意, 粉, ，, 搞, 两, 粒, 鸡, 蛋, ，, r, u, n, n...","親, 要, 食, 意, 粉, ，, 攪, 兩, 碌, 雞, 蛋, ，, r, a, i, n...",42,12,1,2,15,./audio/1289.wav
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,495,我個Favourite嘅band係‘BTS’，我每次聽佢地嘅歌曲都感覺好fresh，而且佢地...,我個 favourite 嘅 band 系 B T S，我每次聽佢哋啲歌曲都感覺好 fres...,0.3729,0.7273,0.3591,6.421,2.3061,"我個, Favourite, 嘅, band, 係‘, BTS, ’，我每次聽佢地嘅歌曲都感...","我個, favourite, 嘅, band, 系, B, T, S, ，我每次聽佢哋啲歌曲...",...,0,8,"我, 個, F, a, v, o, u, r, i, t, e, 嘅, b, a, n, d...","我, 個, , f, a, v, o, u, r, i, t, e, , 嘅, , b...",59,9,6,7,22,./audio/765.wav
496,496,無論讀幾多書，都係experience會教曬你，唔好怕explore新嘅地方和了解新嘅事物。,無論讀幾多書，都係experience會教授你，唔好派 explore 新嘅地方同埋了解新嘅事物。,0.1304,0.4000,0.2935,5.909,1.7346,"無論讀幾多書，都係, experience, 會教曬你，唔好怕, explore, 新嘅地方...","無論讀幾多書，都係, experience, 會教授你，唔好派, explore, 新嘅地方...",...,0,2,"無, 論, 讀, 幾, 多, 書, ，, 都, 係, e, x, p, e, r, i, e...","無, 論, 讀, 幾, 多, 書, ，, 都, 係, e, x, p, e, r, i, e...",46,3,0,3,6,./audio/1356.wav
497,497,我哋嘅長者搵唔到工作，政府需要提供更多支援，包括job training programs。,我哋嘅長者揾唔到工作，政府需要提供更多支援，包括job training programmes。,0.0652,0.4000,0.2555,5.461,1.3953,"我哋嘅長者搵唔到工作，政府需要提供更多支援，包括, job, training, progr...","我哋嘅長者揾唔到工作，政府需要提供更多支援，包括, job, training, progr...",...,0,2,"我, 哋, 嘅, 長, 者, 搵, 唔, 到, 工, 作, ，, 政, 府, 需, 要, 提...","我, 哋, 嘅, 長, 者, 揾, 唔, 到, 工, 作, ，, 政, 府, 需, 要, 提...",46,1,0,2,3,./audio/408.wav
498,498,去Central嘅IFC買Burberry嘅新款手袋，sales話佢地仲有折扣呢!,去 central 嘅 IFC，買 Burberry 嘅新款手袋， sell 話佢哋仲有折扣呢～,0.3171,0.4444,0.1304,13.355,1.7409,"去, Central, 嘅, IFC, 買, Burberry, 嘅新款手袋，, sales...","去, central, 嘅, IFC, ，買, Burberry, 嘅新款手袋，, sell...",...,0,4,"去, C, e, n, t, r, a, l, 嘅, I, F, C, 買, B, u, r...","去, , c, e, n, t, r, a, l, , 嘅, , I, F, C, ，...",41,6,0,7,13,./audio/1614.wav


In [83]:
playAudio(evalResultDF.iloc[6]["audio"])
print(evalResultDF.iloc[6]["ref_words_array"])
print(evalResultDF.iloc[6]["pred_words_array"])

最近迷上飲, green, tea, latte, ，一, sec, 係清新嘅, green, tea, 味，下一, sec, 又係, milky, creamy, textures, ，真係好, enjoy, !
最近迷想飲, Green, Tea, 綠, tea, 一塞係色新嘅, Green, Tea, 味，下一塞又係, Milky, Creamy, Texture, ，可真係好人中意。


In [84]:
evalResultDF.iloc[6]

index                                                                 6
ref_text              最近迷上飲green tea latte，一sec係清新嘅green tea味，下一sec又...
pred_text             最近迷想飲Green Tea綠tea一塞係色新嘅Green Tea味，下一塞又係Milky ...
cer                                                               0.359
wer                                                              0.9444
rtf                                                              0.2774
audio_duration_sec                                                7.765
inference_time_sec                                               2.1538
ref_words_array       最近迷上飲, green, tea, latte, ，一, sec, 係清新嘅, green...
pred_words_array      最近迷想飲, Green, Tea, 綠, tea, 一塞係色新嘅, Green, Tea,...
ref_words                                                            18
wer_substitutions                                                    12
wer_deletions                                                         5
wer_insertions                                                  